# FLARE — Adversarial Cross-Examination for Evacuee-Tweet Relevance

*FLARE — Foundation-model Labelling via Adversarial Review & Escalation*

Two independent LLMs (Claude, GPT) each label whether a tweet describes the *author's own* wildfire evacuation, then cross-examine each other's reasoning before a verdict is finalized. Disagreement routes the tweet to a human labeller rather than being averaged away.

Real tweets from the Kincade, Getty, and Tick fire datasets.

## Setup

Model verdicts on ambiguous posts vary between calls (neither model accepts a temperature override). Cases 1–4 replay real captured transcripts from `demo_cache/pinned/`; the bonus cell at the end calls both APIs live.

In [ ]:
from ace_debate import ACEDebate, DebateOutcome
from IPython.display import Markdown, display

engine = ACEDebate()

OUTCOME_BADGE = {
    "agree":     "✅ **AGREE**",
    "converged": "⚠️ **CONVERGED**",
    "disagree":  "🔺 **DISAGREE — escalated to human labeller**",
}

def display_result(result):
    lines = []
    lines.append("**Round 1 — independent verdicts**")
    lines.append(f"- Claude: `{result.round1_claude.verdict.value}` — {result.round1_claude.reasoning}")
    lines.append(f"- GPT: `{result.round1_gpt.verdict.value}` — {result.round1_gpt.reasoning}")

    if result.outcome != DebateOutcome.AGREE:
        lines.append("")
        lines.append("**Round 2 — adversarial flaw-finding**")
        lines.append("- Claude found in GPT's argument:")
        for f in result.round2_claude_flaw.flaws_found:
            lines.append(f"  - {f}")
        lines.append("- GPT found in Claude's argument:")
        for f in result.round2_gpt_flaw.flaws_found:
            lines.append(f"  - {f}")
        lines.append("")
        lines.append("**Round 3 — rebuttal**")
        lines.append(f"- Claude: `{result.round3_claude.revised_verdict.value}` (changed mind: {result.round3_claude.changed_mind})")
        lines.append(f"- GPT: `{result.round3_gpt.revised_verdict.value}` (changed mind: {result.round3_gpt.changed_mind})")

    lines.append("")
    badge = OUTCOME_BADGE[result.outcome.value]
    final = result.final_verdict.value if result.final_verdict else "None — human decides"
    lines.append(f"**Outcome:** {badge}  ")
    lines.append(f"**Final verdict:** `{final}`  ")
    lines.append(f"**Confidence:** {result.confidence:.0%}")

    if result.human_escalation_summary:
        lines.append("")
        lines.append("---")
        lines.append("```")
        lines.append(result.human_escalation_summary)
        lines.append("```")

    display(Markdown("\n\n".join(lines)))

## Case 1 — Clear, direct account (Kincade Fire)

First-person, unambiguous. Expect fast agreement.

In [ ]:
post1 = (
    "Honestly I feel like I've aged a month since we evacuated Healdsburg "
    "Saturday morning. #KincadeFire"
)

result1 = engine.debate_from_pinned_transcript(post1, case_id="wf_case1_relevant")
display_result(result1)

*Both models agree in Round 1 — no cross-examination needed.*

## Case 2 — News repost (Getty Fire)

Same surface keywords, but a third-party news broadcast, not a first-person account.

In [ ]:
post2 = (
    'Top story: ABC News: "LIVE: We\'re in Los Angeles as the #GettyFire is '
    'exploding in size and forcing thousands of evacuations." '
    "https://t.co/BAK31hKl3z, see more https://t.co/KN1nIiyqVu"
)

result2 = engine.debate_from_pinned_transcript(post2, case_id="wf_case2_irrelevant")
display_result(result2)

*Both correctly reject it as non-first-person — again, no debate needed.*

## Case 3 — Genuinely ambiguous (Getty Fire)

Author reports a warning and hearing helicopters, but never says they evacuated.

In [ ]:
post3 = (
    "I don't think I've ever heard helicopters for this long of a time "
    "before. They've been up there since I got the first evacuation "
    "warning at 2:30am (and I'm sure before). I can't even imagine the "
    "effort this takes. #GettyFire"
)

result3 = engine.debate_from_pinned_transcript(post3, case_id="wf_case3_ambiguous")
display_result(result3)

*Claude starts at `unsure`, GPT at `relevant`; Claude concedes after cross-examination. Both converge on `relevant`, but at lower confidence — flagged, not silently trusted.*

## Case 4 — Contested (Kincade Fire)

Requests an evacuation map — implies stake, but not confirmed evacuation. Reproducibly split across repeated runs.

In [ ]:
post4 = (
    "#KincadeFire Can someone pleasee DM me an updated evacuation map \U0001F612 "
    "I wish they posted everywhere and constantly updated."
)

result4 = engine.debate_from_pinned_transcript(post4, case_id="wf_case4_contested")
display_result(result4)

*Claude holds `unsure`; GPT moves to `relevant`. Still disagreeing after Round 3 — escalated to a human with both arguments attached, not averaged.*

## Bonus — a live call

Everything above replayed captured transcripts. This cell calls the real Claude + GPT APIs live — edit `live_post` and re-run with any text.

In [ ]:
live_post = (
    "Just got back home after a week at my sister's place in Petaluma. "
    "So relieved the house made it through the #KincadeFire untouched."
)

result_live = engine.debate(live_post, case_id="live_demo_wildfire")
display_result(result_live)